## Chat history with memory

### Load ENV file

In [10]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

print(os.getenv('LANGSMITH_PROJECT'))

EALangchainTrainingsTest


### Create local/cloud LLM object

In [11]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="gemini-3-flash-preview:cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=350,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm_1 = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest ",
    temperature=0.5,
    max_tokens=350
)

ollama_local_llm_2 = ChatOllama(
    base_url="http://localhost:11434/",
    model="qwen2.5:7b",
    temperature=0.5,
    max_tokens=350
)

### Message history with ChatMessageHistory

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [18]:
template = ChatPromptTemplate.from_messages([
    ("placeholder", "{history}"),
    ("human", "{prompt}")
])

chain = template | ollama_cloud_llm | StrOutputParser()

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history")

session_id = "Shashank"

get_session_history(session_id).clear()

response1 = history.invoke(
    {"prompt": "What is the advantage of running the LLM in local machine? Just get me answer in bullet points and sub-bullet points. Keep the answer short."},
    config={"configurable": {"session_id": session_id}})

response2 = history.invoke(
    {"prompt": "How about for cloud?"},
    config={"configurable": {"session_id": session_id}})

print(response1)
print("\n--------------------------------------------------------------------\n")
print(response2)

Here are the primary advantages of running an LLM on a local machine:

*   **Privacy and Security**
    *   **Data Sovereignty:** Your data never leaves your hardware; no third-party access.
    *   **Compliance:** Easier to meet strict regulatory requirements (GDPR, HIPAA).

*   **Cost Efficiency**
    *   **No Subscriptions:** Avoid monthly fees for premium AI services.
    *   **No API Fees:** Zero cost per token; ideal for high-volume processing.

*   **Offline Accessibility**
    *   **No Internet Required:** Works in remote areas or secure, air-gapped environments.
    *   **Reliability:** No downtime caused by server outages or network issues.

*   **Performance and Latency**
    *   **Zero Network Lag:** Faster response times by eliminating data transmission delays.
    *   **Dedicated Resources:** No "peak hour" slowdowns from shared cloud servers.

*   **Customization and Control**
    *   **Model Choice:** Freedom to use specialized, niche, or experimental models.
    *   **

### Message history with SqlChatMessageHistory

In [17]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

template = ChatPromptTemplate.from_messages([
    ("placeholder", "{history}"),
    ("human", "{prompt}")
])

chain = template | ollama_cloud_llm | StrOutputParser()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    return SQLChatMessageHistory(session_id=session_id, connection_string="sqlite:///./chat_history.db")

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history")

session_id = "Shashank"

get_session_history(session_id).clear()

response1 = history.invoke(
    {"prompt": "What is the advantage of running the LLM in local machine? Just get me answer in bullet points and sub-bullet points. Keep the answer short."},
    config={"configurable": {"session_id": session_id}})

response2 = history.invoke(
    {"prompt": "How about for cloud?"},
    config={"configurable": {"session_id": session_id}})

print(response1)
print("\n--------------------------------------------------------------------\n")
print(response2)

*   **Privacy and Security**
    *   Data stays on your hardware; no third-party access.
    *   Ideal for handling sensitive or proprietary information.

*   **Cost Efficiency**
    *   No monthly subscription fees (e.g., ChatGPT Plus).
    *   No per-token API costs for developers.

*   **Offline Accessibility**
    *   Functions without an internet connection.
    *   Reliable for remote work or travel.

*   **Customization and Control**
    *   Ability to use "uncensored" models without provider-imposed guardrails.
    *   Full control over system prompts, parameters, and model versions.

*   **Performance and Latency**
    *   Zero network latency; no waiting for server responses.
    *   Consistent performance regardless of peak usage times on public servers.

*   **Development and Testing**
    *   Unlimited experimentation without worrying about rate limits.
    *   Easier integration into local workflows and private databases (RAG).

-------------------------------------------